# 27 — Observability: watching your agents run

Every track so far taught you to *build* an agent. Production asks a different question: **what is my agent doing right now?** Which tool is slow, which delegation failed, what did the sub-agent actually return at 3 a.m.? Track E gave you two records — the *trace* (the loop's flight recorder) and the *Context* (the window) — but neither carries timing, nesting for humans, or a UI. That third record is the **span tree**: every observability tool, open-source or SaaS, stands on it.

`lab_core.trace` is that idea in ~100 lines with nothing hidden: a `Span` is one timed, named operation; spans nest under a `Tracer`; a `Sink` is the exporter boundary that ships each span to a backend as it opens and closes. Two seams instrument the whole lab: `tracer.wrap_tools(...)` turns any notebook-01 tools dict into a span tree, and `DeepAgent(tracer=...)` spans the agent root and every tool call — including `task` delegations, so sub-agent work shows up as a timed child span.

The industrial half is **`notebooks/obs.py`**, ported from the dspy-lab course's helper of the same name and carrying the same robustness contract: sessions are **dated** (`agentic-lab-YYYY-MM-DD`, so days never collide in a UI), the MLflow store is **pinned to the repo and remnant-proof** (an unusable leftover `mlflow.db` triggers a clean fallback, never an error), and UIs are **found-or-started from a cell** (a healthy server of ours is reused, a stranger's port is skipped, otherwise a background process is started that survives kernel restarts). Where dspy-lab's helpers switch on DSPy autologging, ours provide *sinks* that bridge the lab's own spans into each backend.

No research paper backs this notebook — observability is engineering, not literature. Primary sources: [MLflow tracing](https://mlflow.org/docs/latest/genai/tracing/), [Phoenix tracing](https://arize.com/docs/phoenix/tracing/llm-traces), and [OpenTelemetry traces](https://opentelemetry.io/docs/concepts/signals/traces/).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.patterns import run_tool_loop
from lab_core.trace import Tracer

# The mini-tracer moment: wrap the tools of a notebook-01 loop, get a span tree.
facts = {'react': 'ReAct interleaves reasoning traces with actions.'}
tracer = Tracer()
tools = tracer.wrap_tools({'lookup': lambda term: facts.get(str(term), 'not found')})

def policy(goal, trace):
    if not trace:
        return {'tool': 'lookup', 'input': 'react'}
    return {'final': trace[-1]['observation']}

with tracer.span('loop:react'):
    result = run_tool_loop('Explain ReAct', policy, tools, max_steps=3)

print(tracer.render())
print('\nanswer:', result.answer)

In [ ]:
# The DeepAgent seam: pass a tracer, get the whole run as one tree -
# including the task delegation and a deliberately failing read at the end.
from lab_core.deep import DeepAgent, SubAgent, VirtualFS, compose_system_prompt

fs = VirtualFS()
fs.write('/sources/doc-a.md', 'CLAIM: spans make latency visible.\n' + 'detail. ' * 40)

def read_claims(task, context, files):
    text = files.read(task.split()[-1])
    context.add('tool:read_file', text[:80])
    return {'summary': text.splitlines()[0], 'detail': text}

def policy(goal, todos, trace, context):
    if not todos.items:
        return {'tool': 'write_todos', 'input': {'add': ['digest doc-a', 'check missing file']}}
    pending = todos.next_pending()
    if pending is None:
        return {'final': 'done'}
    index, last = todos.items.index(pending), (trace[-1] if trace else {})
    if pending.text == 'digest doc-a':
        if last.get('tool') == 'task':
            return {'tool': 'write_todos', 'input': {'done': index}}
        return {'tool': 'task', 'input': {'agent': 'reader', 'task': 'digest /sources/doc-a.md'}}
    return {'tool': 'read_file', 'input': {'path': '/missing.md'}}   # the failure we want to SEE

agent_tracer = Tracer()
agent = DeepAgent('reviewer', system_prompt=compose_system_prompt('You review sources.'),
                  policy=policy, fs=fs, tracer=agent_tracer)
agent.subagents = {'reader': SubAgent('reader', 'Extract CLAIM lines.', read_claims, fs)}

result = agent.run('Digest the source', run_id='obs-1')
print(agent_tracer.render())
print('\nloop ended with:', result.error)

> **What you should see:** one `agent:reviewer` root span with a `tool:` child per step — the `tool:task` span timing the whole sub-agent delegation — and a final `tool:read_file` span marked `!! KeyError: ...` where the missing file blew up. The loop *result* only says a tool failed; the span tree says **which call, how deep, after how long** — that is the entire pitch of observability, in one print.

The two cells below ship the very same spans to real UIs. Both run locally with **no API key and no signup**; they are gated only on the optional installs (`pip install -e '.[obs]'`).

In [ ]:
# Option A - MLflow: log spans to the dated experiment, then open the UI from this cell.
from importlib.util import find_spec

if find_spec('mlflow'):
    import obs
    obs.enable_mlflow()
    ml_tracer = Tracer(sinks=[obs.mlflow_sink()])
    agent.tracer = ml_tracer
    agent.run('Digest the source', run_id='obs-2')
    print(f'{len(ml_tracer.roots[0].children)} tool spans mirrored into MLflow')
    obs.mlflow_ui()          # find-or-start; click the printed link, open the Traces tab
else:
    print("Install the optional extra to run this cell: pip install -e '.[obs]'")

In [ ]:
# Option B - Arize Phoenix: a purpose-built trace UI, fed over OpenTelemetry.
if find_spec('phoenix'):
    import obs
    if obs.enable_phoenix() is not None:     # find-or-start the server, then route spans
        px_tracer = Tracer(sinks=[obs.phoenix_sink()])
        agent.tracer = px_tracer
        agent.run('Digest the source', run_id='obs-3')
        print(f'{len(px_tracer.roots[0].children)} tool spans shipped to Phoenix')
else:
    print("Install the optional extra to run this cell: pip install -e '.[obs]'")

> **In the UIs:** MLflow's Traces tab and Phoenix's project view each show the run as the same tree cell 2 printed — `agent:reviewer` → tool spans with inputs, observation sizes, latencies, and the error span highlighted. Phoenix receives spans over the network, so its server must be up before the traced cells run — which is why `enable_phoenix()` launches before it routes. If obs reports it could not find or start Phoenix (rare — a locked-down environment, or a Phoenix release incompatible with your exact Python patch version), the cell degrades to a printed notice and nothing else breaks; run `phoenix serve` from a terminal and rerun, or stay with MLflow.

**The landscape, and the choice.**

| Tool | Runs where | Fed by | Best at |
| --- | --- | --- | --- |
| `Tracer.render()` | in the notebook | the spans themselves | zero-setup debugging, tests |
| **MLflow** | local, free | `obs.mlflow_sink()` | traces plus experiment/run tracking |
| **Phoenix** | local, free | `obs.phoenix_sink()` over OTel | the nicest trace-inspection UI |
| SaaS (Braintrust, LangSmith, ...) | hosted, API key | OTel exporters | teams already on those platforms |

Because the `Sink` protocol is two methods, pointing the lab at any OTel-speaking backend is one more sink — the same portability argument as Track D, applied to telemetry. Production caveats stated once: the sinks assume single-threaded spans (notebook 02's `fan_out_fan_in` would need context propagation, which is exactly what OpenTelemetry's context API adds), and span *attributes* can carry sensitive text — cap and scrub them the way notebook 23 capped observations. The dspy-lab course teaches this same machinery against DSPy programs in its notebook 11; the two `obs.py` files are deliberately twins.